In [1]:
!pip install -q gensim

import os

os.makedirs("models", exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.9 MB/s eta 0:00:00


In [2]:
%%writefile train_lda.py
import os
import json

from sklearn.datasets import fetch_20newsgroups

from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora, models


def preprocess(text):
    """
    Preprocess a single document:
    - Tokenize
    - Lowercase
    - Remove stopwords
    - Keep words with length >= 3
      (simple_preprocess with min_len=3 already does this)
    """
    tokens = simple_preprocess(text, deacc=True, min_len=3)
    tokens = [token for token in tokens if token not in STOPWORDS]
    return tokens


def main():
    os.makedirs("models", exist_ok=True)

    print("Loading 20 Newsgroups dataset (train subset, first 1000 docs)...")
    newsgroups_train = fetch_20newsgroups(
        subset='train',
        remove=('headers', 'footers', 'quotes')
    )

    raw_docs = newsgroups_train.data[:1000]
    print(f"Number of documents used: {len(raw_docs)}")

    print("Preprocessing documents...")
    processed_docs = [preprocess(doc) for doc in raw_docs]

    print("Creating dictionary...")
    dictionary = corpora.Dictionary(processed_docs)

    dictionary.filter_extremes(no_below=5, no_above=0.5)

    print("Converting documents to Bag-of-Words representation...")
    corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

    print("Training LDA model with 10 topics...")
    lda_model = models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=10,
        passes=15,
        alpha='auto',
        eta='auto',
        random_state=42
    )

    model_path = os.path.join("models", "lda_model.model")
    dict_path = os.path.join("models", "lda_dictionary.dict")

    print(f"Saving LDA model to: {model_path}")
    lda_model.save(model_path)

    print(f"Saving dictionary to: {dict_path}")
    dictionary.save(dict_path)

    print("\nDiscovered Topics (top 15 words):")
    topics = lda_model.print_topics(num_topics=10, num_words=15)
    for topic_id, topic_words in topics:
        print(f"\nTopic {topic_id}:")
        print(topic_words)


if __name__ == "__main__":
    main()


Writing train_lda.py


In [3]:
!python train_lda.py


Loading 20 Newsgroups dataset (train subset, first 1000 docs)...
Number of documents used: 1000
Preprocessing documents...
Creating dictionary...
Converting documents to Bag-of-Words representation...
Training LDA model with 10 topics...
Saving LDA model to: models/lda_model.model
Saving dictionary to: models/lda_dictionary.dict

Discovered Topics (top 15 words):

Topic 0:
0.146*"max" + 0.022*"period" + 0.016*"play" + 0.014*"power" + 0.008*"second" + 0.008*"vancouver" + 0.007*"motif" + 0.007*"mhz" + 0.007*"louis" + 0.006*"scoring" + 0.006*"chicago" + 0.006*"like" + 0.006*"pittsburgh" + 0.006*"detroit" + 0.006*"type"

Topic 1:
0.015*"know" + 0.008*"person" + 0.007*"people" + 0.007*"problem" + 0.007*"windows" + 0.006*"like" + 0.006*"software" + 0.006*"moral" + 0.006*"question" + 0.006*"yes" + 0.005*"code" + 0.005*"source" + 0.005*"known" + 0.005*"wrong" + 0.005*"cable"

Topic 2:
0.012*"use" + 0.009*"problem" + 0.008*"program" + 0.008*"want" + 0.007*"files" + 0.007*"like" + 0.007*"disk" +

In [4]:
%%writefile label_topics.py
import os
import json

from gensim import models


def load_model_and_dictionary(model_dir="models"):
    model_path = os.path.join(model_dir, "lda_model.model")

    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"LDA model not found at {model_path}. "
            "Make sure you have run train_lda.py first."
        )

    print(f"Loading LDA model from: {model_path}")
    lda_model = models.LdaModel.load(model_path)

    return lda_model


def main():
    os.makedirs("models", exist_ok=True)

    lda_model = load_model_and_dictionary("models")

    num_topics = lda_model.num_topics
    print(f"\nNumber of topics in the model: {num_topics}\n")

    topic_labels = {}

    topics = lda_model.show_topics(
        num_topics=num_topics,
        num_words=20,
        formatted=False
    )

    for topic_id, word_probs in topics:
        print("=" * 80)
        print(f"Topic {topic_id} - Top 20 words with probabilities:")
        for word, prob in word_probs:
            print(f"{word:15s}  {prob:.4f}")
        print("-" * 80)
        label = input(
            f"Enter a label/name for Topic {topic_id} "
            f"(press Enter to keep default 'Topic {topic_id}'): "
        ).strip()

        if label == "":
            label = f"Topic {topic_id}"

        topic_labels[str(topic_id)] = label

    labels_path = os.path.join("models", "topic_labels.json")
    with open(labels_path, "w") as f:
        json.dump(topic_labels, f, indent=2)

    print(f"\nTopic labels saved to: {labels_path}\n")

    print("Final summary of topics and their labels:")
    for topic_id, label in topic_labels.items():
        print(f"Topic {topic_id}: {label}")


if __name__ == "__main__":
    main()


Writing label_topics.py


In [5]:
!python label_topics.py


Loading LDA model from: models/lda_model.model

Number of topics in the model: 10

Topic 0 - Top 20 words with probabilities:
max              0.1464
period           0.0216
play             0.0163
power            0.0143
second           0.0081
vancouver        0.0081
motif            0.0073
mhz              0.0065
louis            0.0065
scoring          0.0065
chicago          0.0065
like             0.0063
pittsburgh       0.0062
detroit          0.0060
type             0.0056
jose             0.0055
san              0.0053
los              0.0053
time             0.0052
scope            0.0049
--------------------------------------------------------------------------------
Enter a label/name for Topic 0 (press Enter to keep default 'Topic 0'): Game
Topic 1 - Top 20 words with probabilities:
know             0.0147
person           0.0079
people           0.0073
problem          0.0073
windows          0.0069
like             0.0061
software         0.0059
moral            0.0058
q

In [6]:
%%writefile infer_topics.py
import os
import json
from textwrap import shorten

from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS


def preprocess(text):
    """
    Same preprocessing as training:
    - Tokenize
    - Lowercase
    - Remove stopwords
    - Keep words with length >= 3
    """
    tokens = simple_preprocess(text, deacc=True, min_len=3)
    tokens = [token for token in tokens if token not in STOPWORDS]
    return tokens


def load_dictionary_and_model(model_dir="models"):
    dict_path = os.path.join(model_dir, "lda_dictionary.dict")
    model_path = os.path.join(model_dir, "lda_model.model")

    if not os.path.exists(dict_path):
        raise FileNotFoundError(
            f"Dictionary not found at {dict_path}. "
            "Make sure you have run train_lda.py first."
        )
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"LDA model not found at {model_path}. "
            "Make sure you have run train_lda.py first."
        )

    print(f"Loading dictionary from: {dict_path}")
    dictionary = corpora.Dictionary.load(dict_path)

    print(f"Loading LDA model from: {model_path}")
    lda_model = models.LdaModel.load(model_path)

    return dictionary, lda_model


def load_topic_labels(model_dir="models"):
    labels_path = os.path.join(model_dir, "topic_labels.json")
    if os.path.exists(labels_path):
        with open(labels_path, "r") as f:
            labels = json.load(f)
        print(f"Loaded topic labels from: {labels_path}")
    else:
        print("Warning: topic_labels.json not found. Using default labels.")
        labels = {}
    return labels


def get_topic_name(topic_id, labels):
    return labels.get(str(topic_id), f"Topic {topic_id}")


def classify_document(text, dictionary, lda_model, labels, topn=3):
    """
    Classify a single text document:
    - Preprocess
    - Convert to BoW
    - Get topic distribution
    - Return top n topics with probabilities
    """
    tokens = preprocess(text)
    bow = dictionary.doc2bow(tokens)

    topic_dist = lda_model.get_document_topics(bow, minimum_probability=0.0)

    topic_dist = sorted(topic_dist, key=lambda x: x[1], reverse=True)

    top_topics = topic_dist[:topn]

    results = []
    for topic_id, prob in top_topics:
        name = get_topic_name(topic_id, labels)
        top_words = lda_model.show_topic(topic_id, topn=5)
        top_words_only = [w for w, p in top_words]
        results.append({
            "topic_id": topic_id,
            "topic_name": name,
            "probability": prob,
            "top_words": top_words_only
        })

    return results


def print_topic_summary(lda_model, labels, topn=10):
    """
    Print summary of all topics: label + top words.
    """
    print("\n=== Topic Summary ===")
    topics = lda_model.show_topics(
        num_topics=lda_model.num_topics,
        num_words=topn,
        formatted=False
    )
    for topic_id, word_probs in topics:
        label = get_topic_name(topic_id, labels)
        words = [w for w, p in word_probs]
        print(f"\nTopic {topic_id} - {label}")
        print("Top words:", ", ".join(words))
    print("=====================\n")


def main():
    dictionary, lda_model = load_dictionary_and_model("models")
    labels = load_topic_labels("models")

    print_topic_summary(lda_model, labels, topn=10)

    samples = [
        "The new graphics card delivers amazing performance for gaming. "
        "The GPU can handle 4K resolution easily with ray tracing enabled. "
        "Gamers will love the improved frame rates.",

        "Scientists discovered a new exoplanet orbiting a distant star in the "
        "habitable zone. The research team published their findings in Nature "
        "journal. This discovery could provide insights into planetary formation.",

        "The basketball team won the championship after an incredible final game. "
        "The players celebrated with fans in the stadium. It was the team's first "
        "title in twenty years.",

        "Congress passed a new bill regarding healthcare reform. The president is "
        "expected to sign the legislation next week. The policy will affect millions "
        "of citizens across the country.",

        "I love cooking Italian food at home. Pasta carbonara and margherita pizza "
        "are my favorite dishes to make. Fresh ingredients make all the difference "
        "in authentic recipes."
    ]

    print("Running classification on 5 sample documents...\n")

    for i, doc in enumerate(samples, start=1):
        print("=" * 80)
        print(f"Sample {i}:")
        preview = shorten(doc, width=200, placeholder="...")
        print("Document preview:")
        print(preview)
        print("-" * 80)

        results = classify_document(doc, dictionary, lda_model, labels, topn=3)

        for res in results:
            print(
                f"Topic {res['topic_id']} - {res['topic_name']}: "
                f"prob={res['probability']:.4f}"
            )
            print("  Top words:", ", ".join(res["top_words"]))
        print("=" * 80 + "\n")

    print("You can now enter your own text for classification.")
    print("Type 'quit' on a blank line to exit.\n")
    while True:
        user_text = input("Enter a document (or 'quit' to stop): ").strip()
        if user_text.lower() == "quit":
            break
        if not user_text:
            continue

        print("\nDocument preview:")
        print(shorten(user_text, width=200, placeholder="..."))
        print("-" * 80)

        results = classify_document(user_text, dictionary, lda_model, labels, topn=3)
        for res in results:
            print(
                f"Topic {res['topic_id']} - {res['topic_name']}: "
                f"prob={res['probability']:.4f}"
            )
            print("  Top words:", ", ".join(res["top_words"]))
        print("=" * 80 + "\n")


if __name__ == "__main__":
    main()


Writing infer_topics.py


In [7]:
!python infer_topics.py


Loading dictionary from: models/lda_dictionary.dict
Loading LDA model from: models/lda_model.model
Loaded topic labels from: models/topic_labels.json

=== Topic Summary ===

Topic 0 - Game
Top words: max, period, play, power, second, vancouver, motif, mhz, louis, scoring

Topic 1 - city
Top words: know, person, people, problem, windows, like, software, moral, question, yes

Topic 2 - computer
Top words: use, problem, program, want, files, like, disk, time, thanks, need

Topic 3 - church
Top words: jesus, god, people, think, matthew, know, time, good, man, things

Topic 4 - sport
Top words: use, health, father, year, years, son, spirit, state, car, medical

Topic 5 - PC
Top words: space, nasa, shuttle, use, windows, like, data, com, scsi, chip

Topic 6 - mixed
Top words: good, excellent, new, think, missing, know, year, israel, like, cover

Topic 7 - random gorup
Top words: edu, thanks, like, know, mail, card, sale, video, people, time

Topic 8 - war
Top words: people, armenian, armenia